# Layerwise embedding kNN with TabPFN v2.6

This notebook runs the embedding-based kNN experiment on four TabArena
classification datasets: **anneal**, **blood-transfusion**, **credit-g**, and
**diabetes**.

For each dataset:

1. Fit TabPFN v2.6 with one ensemble member.
2. Capture the input embedding (`before_tf`) and all 24 block outputs with
   forward hooks.
3. Build one feature vector per row: feature tokens only, the target column is
   excluded, and the 64 thinking rows are removed from block outputs.
4. Run kNN at every representation (`k = 1, 3, 5, 10`) and compare it with
   input-space kNN, TabPFN's own prediction, and the majority-class baseline.

The implementation lives in `embedding_knn.py`; this notebook calls it step by
step. For the command-line experiment, run `python embedding_knn.py`.


## 1. Setup

`MAX_ROWS = 500` keeps the CPU run and the activation files small. Set
`MAX_ROWS = None` (or pass `--max-rows 0` to the script) to use the full
datasets.


In [ ]:
from pathlib import Path

import pandas as pd

from embedding_knn import (
    DATASETS,
    MAX_ROWS,
    RESULTS_DIR,
    build_summary,
    evaluate_one,
    extract_one,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("datasets:", list(DATASETS))
print("max rows:", MAX_ROWS)


## 2. Extract embeddings

Each dataset is processed independently. `extract_one` runs the classifier
once, lets TabPFN perform its own preprocessing, and records the tensors that
enter the architecture through forward hooks.


In [ ]:
infos = []

for name, target in DATASETS.items():
    info = extract_one(name, target, max_rows=MAX_ROWS, seed=0)
    infos.append(info)

pd.DataFrame(infos)


## 3. Evaluate kNN at every representation

`evaluate_one` loads the saved activations and evaluates:

- `input`: kNN on the preprocessed input features;
- `before_tf`: kNN on the input embedding;
- `L0` ... `L23`: kNN on each transformer block output.

All kNN models use standardized features and the same train/test split.


In [ ]:
rows = []

for name in DATASETS:
    rows.extend(evaluate_one(name))

results = pd.DataFrame(rows)
results_path = RESULTS_DIR / "embedding_knn_all.csv"
results.to_csv(results_path, index=False)

summary = build_summary(results)
summary_path = RESULTS_DIR / "summary.csv"
summary.to_csv(summary_path, index=False)

print("results:", results.shape)
print("saved:", results_path)
print("saved:", summary_path)


## 4. Results (k = 5)

The included results use 500-row stratified subsets (350 train / 150 test),
seed 0. `TabPFN` is the model's own accuracy. `Best block` is the block with
the highest kNN accuracy; `Best agreement` is the block whose kNN predictions
agree most with TabPFN.

| Dataset | TabPFN | Input kNN | before_tf | Best block | Best agreement |
|---|---|---|---|---|---|
| Anneal | 0.980 | 0.880 | 0.907 | L14: 0.980 | L16: 0.980 |
| Blood transfusion | 0.800 | 0.747 | 0.767 | L12: 0.807 | L20: 0.947 |
| Credit-g | 0.713 | 0.660 | 0.647 | L10: 0.733 | L20: 0.840 |
| Diabetes | 0.760 | 0.707 | 0.693 | L20: 0.747 | L20: 0.853 |

Two observations:

- Late blocks are more kNN-friendly than early blocks.
- Accuracy and agreement can peak at different blocks. Similar accuracy does
  not imply the same prediction mechanism.

